# Dubois surface inversion: Python vs C GUI on Winnip C3 data

This notebook loads the precomputed C GUI outputs from `/data/psp/winnip_cpsp/C3`, runs the Python Dubois surface inversion on the same C3 input, and compares the two results numerically.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import numpy as np
import xarray as xr
from dask.diagnostics import ProgressBar

from polsarpro.dev.io import read_C3_psp, read_psp_bin
from polsarpro.dev.metrics import summarize_metrics, visualize_errors
from polsarpro.physical_inversion import dubois_surface_inversion

# Same Dubois parameters as the original notebook.
freq_ghz = 1.2575
thresh1_db = 0
thresh2_db = 0
calibration_coeff = 1.0

input_test_dir = Path("/data/psp/winnip_cpsp/C3")
incidence_angle_file = Path("/data/psp/winnip_cpsp/incidence_angle_ml5x5_rad.bin")
file_out = Path("/data/psp/res/test_dubois_surface_inversion_winnip_python.nc")
file_out.parent.mkdir(parents=True, exist_ok=True)

print(f"Using Dubois frequency: {freq_ghz:.6f} GHz")


## Load C3 input and C GUI reference outputs

In [ ]:
C3 = read_C3_psp(input_test_dir)
row_dim, col_dim = C3.dims

incidence_angle_values = np.fromfile(
    incidence_angle_file,
    dtype="<f4",
    count=C3.sizes[row_dim] * C3.sizes[col_dim],
)
incidence_angle_values[incidence_angle_values == -10000] = np.nan
incidence_angle = xr.DataArray(
    incidence_angle_values.reshape(C3.sizes[row_dim], C3.sizes[col_dim]),
    dims=(row_dim, col_dim),
    coords={row_dim: C3.coords[row_dim], col_dim: C3.coords[col_dim]},
    name="incidence_angle",
)

out_c = {}
for name in (
    "dubois_er",
    "dubois_mv",
    "dubois_ks",
    "dubois_mask_out",
    "dubois_mask_in",
    "dubois_mask_valid_in_out",
):
    out_c[name] = read_psp_bin(input_test_dir / f"{name}.bin")

print(f"Loaded C3 input with shape {C3.sizes[row_dim]} x {C3.sizes[col_dim]}")


## Incidence Angle

Plot the multilooked incidence angle raster used by both the Python and C GUI runs.


In [ ]:
import matplotlib.pyplot as plt

incidence_angle_fig = incidence_angle.transpose(*reversed(incidence_angle.dims))

plt.figure(figsize=(10, 5))
plt.imshow(incidence_angle_fig, cmap="viridis", interpolation="none")
plt.title("Incidence angle (transposed for display)")
plt.axis("off")
plt.colorbar(fraction=0.046, pad=0.04, location="bottom")
plt.tight_layout()


## Run the Python inversion

In [ ]:
if file_out.exists():
    file_out.unlink()

with ProgressBar():
    out_py = dubois_surface_inversion(
        C3,
        incidence_angle=incidence_angle,
        freq_ghz=freq_ghz,
        thresh1=thresh1_db,
        thresh2=thresh2_db,
        calibration_coeff=calibration_coeff,
    )
    out_py.to_netcdf(file_out)


## Compare Python and C GUI outputs

In [ ]:
df = summarize_metrics(out_py, out_c, short_titles=False, verbose=False)
df

In [ ]:
# Use transposed views for plotting so x and y are swapped in the figures.
out_py_fig = out_py.transpose("x", "y")
out_c_fig = {name: arr.T for name, arr in out_c.items()}

visualize_errors(out_py=out_py_fig, out_c=out_c_fig, sub_az=1, sub_rg=1)
